In [1]:
import json
import os
import numpy as np
import strawberryfields as sf
from strawberryfields.ops import S2gate, BSgate, MeasureHomodyne


def symplectic_form(n_modes: int) -> np.ndarray:
    """Symplectic form in xp ordering: (x1,...,xn,p1,...,pn)."""
    I = np.eye(n_modes)
    O = np.zeros((n_modes, n_modes))
    return np.block([[O, I], [-I, O]])


def smallest_pt_symplectic_eigenvalue(V: np.ndarray) -> float:
    """
    Smallest symplectic eigenvalue of the partially transposed
    2-mode covariance matrix V in xp ordering.
    """
    # Partial transpose on second mode flips p2 -> -p2
    Lambda = np.diag([1.0, 1.0, 1.0, -1.0])
    V_pt = Lambda @ V @ Lambda

    Omega = symplectic_form(2)
    eigvals = np.linalg.eigvals(1j * Omega @ V_pt)
    nus = np.sort(np.abs(eigvals))

    # Symplectic eigenvalues appear in duplicated pairs
    # Return the smallest positive one
    return float(nus[0].real)


def log_negativity_from_cov(V: np.ndarray) -> float:
    """
    Log-negativity for a 2-mode Gaussian state in Strawberry Fields default units.
    With SF default conventions, entanglement is present when nu_tilde < 1.
    """
    nu_tilde = smallest_pt_symplectic_eigenvalue(V)
    return float(max(0.0, -np.log(nu_tilde)))


def main():
    os.makedirs("results", exist_ok=True)

    # Four-mode CV entanglement swapping
    # Modes: 0-1 are one TMS pair, 2-3 are another TMS pair.
    # Middle modes 1 and 2 interfere, then are homodyned.
    r = 0.6  # modest squeezing for stable local simulation

    prog = sf.Program(4)

    with prog.context as q:
        # Two entangled Gaussian resources
        S2gate(r, 0.0) | (q[0], q[1])
        S2gate(r, 0.0) | (q[2], q[3])

        # Middle-mode interference for CV swapping
        BSgate(np.pi / 4, 0.0) | (q[1], q[2])

        # Homodyne postselection on middle modes
        # Using selected value 0.0 keeps the benchmark deterministic and minimal.
        MeasureHomodyne(0.0, select=0.0) | q[1]          # X homodyne
        MeasureHomodyne(np.pi / 2, select=0.0) | q[2]    # P homodyne

    eng = sf.Engine("gaussian")
    result = eng.run(prog)
    state = result.state

    # Reduced Gaussian state on end modes 0 and 3
    mu_03, V_03 = state.reduced_gaussian([0, 3])

    # Native CV entanglement metric
    log_negativity = log_negativity_from_cov(np.array(V_03, dtype=float))

    success_probability = 1.0
    generalized_entanglement_yield = success_probability * log_negativity

    output = {
        "stack": "strawberryfields_cv",
        "paradigm": "CV",
        "protocol": "two_resource_cv_entanglement_swapping",
        "success_probability": success_probability,
        "native_entanglement_metric": log_negativity,
        "generalized_entanglement_yield": generalized_entanglement_yield,
        "notes": (
            "Local Gaussian-backend CV swapping using two TMS resources, "
            "middle-mode beamsplitter interference, and homodyne postselection."
        ),
    }

    with open("results/strawberryfields_cv.json", "w", encoding="utf-8") as f:
        json.dump(output, f, indent=2)

    print(json.dumps(output, indent=2))


if __name__ == "__main__":
    main()

/home/bkiefer/miniconda3/envs/sf-kernel/lib/python3.10/site-packages/strawberryfields/apps/data/sample.py:20: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


{
  "stack": "strawberryfields_cv",
  "paradigm": "CV",
  "protocol": "two_resource_cv_entanglement_swapping",
  "success_probability": 1.0,
  "native_entanglement_metric": 0.5936889212592296,
  "generalized_entanglement_yield": 0.5936889212592296,
  "notes": "Local Gaussian-backend CV swapping using two TMS resources, middle-mode beamsplitter interference, and homodyne postselection."
}
